In [ ]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
import pyarrow.parquet as pq
from tabulate import tabulate

In [ ]:
tabler=pq.read_table("clean_auto_data.parquet")
df=tabler.to_pandas()

In [ ]:
load_dotenv()
phost=os.getenv("DB_HOST")
pport=os.getenv("DB_PORT")
puser=os.getenv("DB_USER")
ppass=os.getenv("DB_PASS")
pname=os.getenv("DB_NAME")

In [ ]:
db_url=f"postgresql+psycopg://{puser}:{ppass}@{phost}:{pport}/{pname}"

In [ ]:
engine=create_engine(url=db_url, echo=False, pool_size=5, max_overflow=10)

In [ ]:
df.to_sql("cars", engine, if_exists="replace", index=False)

In [ ]:
df=pd.read_sql_query('SELECT * FROM cars', con=engine)

In [ ]:
with engine.begin() as conn:
    query="""
        WITH market_benchmarks AS(
            SELECT 
                brand, model, year, price, mileage,
                COUNT(*) OVER(PARTITION BY cars.brand, cars.model, cars.year) AS group_count,
                AVG(cars.price) OVER(PARTITION BY cars.brand, cars.model, cars.year) AS avg_price,
                AVG(cars.mileage) OVER(PARTITION BY cars.brand, cars.model, cars.year) AS avg_mileage
            FROM cars
            WHERE model!='unknown'
        )
        SELECT brand, model, year, price, ROUND(avg_price, 2) as avg_price, ROUND(avg_mileage, 2) AS avg_mileage, group_count, ROUND(avg_price - price, 2) AS byn_profit, ROUND((avg_price - price)*100/avg_price, 2) AS precent_profit  FROM market_benchmarks WHERE group_count>=5 AND price<avg_price*0.8 ORDER BY byn_profit DESC;
    """
    res=conn.execute(text(query))
    print(tabulate(res.fetchall(), headers=res.keys(), tablefmt="psql"))

    # --- Example output in Jupyter Notebook ---
    
    # df_res=pd.read_sql(text(query), con=engine)
    # display(df_res)

    # ------------------------------------------